# CNN ENTRENAMIENTO

### Callback de Accuracy (Keras)

In [ ]:
from dataclasses import dataclass
from datetime import datetime

@dataclass
class MetricsValue:
    value: float
    value_name: str

@dataclass
class MetricsDto:
    name: str
    timestamp: datetime
    step: int
    step_type: str
    phase: str
    values: list[MetricsValue]

In [ ]:
import tensorflow as tf
import json
import paho.mqtt.client as mqtt
import csv
import os
from datetime import datetime
from dataclasses import asdict

folder = "AccuracyFiles"
os.makedirs(folder, exist_ok=True)

filename = folder + "/Accuracy_" + datetime.now().isoformat() + ".csv"

class AccuracyCallback(tf.keras.callbacks.Callback):
    def __init__(self, train_name, epoch_offset=0):
        self.offset = epoch_offset
        self.name = train_name
        try:
            self.mqtt_client = mqtt.Client()
            broker = "localhost"
            port = 1883
            self.mqtt_client.connect(broker, port)
            self.mqtt_client.loop_start()

        except Exception as e:
            print(f"Error conectando con el broker: {e}")
            self.mqtt = None

    def on_epoch_end(self, epoch, logs=None,  ):
        now = datetime.now().isoformat()
        train_dto = MetricsDto(
            name= self.name,
            timestamp=now,
            step=epoch + self.offset,
            step_type="epoch",
            phase="train",
            values=[
                MetricsValue(value=float(logs.get("accuracy", 0.0)), value_name="accuracy"),
                MetricsValue(value=float(logs.get("loss", 0.0)), value_name="loss (scc)")
            ]
        )
        val_dto = MetricsDto(
        name= self.name,
        timestamp=now,
        step=epoch + self.offset,
        step_type="epoch",
        phase="val",
        values=[
            MetricsValue(value=float(logs.get("val_accuracy", 0.0)), value_name="accuracy"),
            MetricsValue(value=float(logs.get("val_loss", 0.0)), value_name="loss (scc)")
        ]
        )
        self.mqtt_client.publish("training/metrics", json.dumps(asdict(train_dto)), qos=1)
        self.mqtt_client.publish("training/metrics", json.dumps(asdict(val_dto)), qos=1)


    def save_data(self, data):
        with open(filename, "a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([datetime.now().isoformat(), data["epoch"], data["accuracy"]])




### 1.1 MobileNetV2 con TL

In [12]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2 
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


layers = tf.keras.layers
models = tf.keras.models
utils = tf.keras.utils

directory = 'datasets/realwaste-main/RealWaste/'

train_dataset = utils.image_dataset_from_directory(
    directory,
    labels="inferred",
    class_names=None,
    subset="training",
    validation_split=0.2,
    batch_size=32,
    image_size=(224,224),
    shuffle=True,
    seed=12,
)

validation_dataset = utils.image_dataset_from_directory(
    directory,
    labels="inferred",
    class_names=None,
    subset="validation",
    validation_split=0.2,
    batch_size=32,
    image_size=(224,224),
    shuffle=True,
    seed=12,
)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
])

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x , training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(9, activation="softmax")(x) 
model = models.Model(inputs, outputs)

model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy', # Sparce porque mis labels son enteros, sin sparce devuelve probabilidades
                  metrics=['accuracy'])

# Entrenar última capa
print("Starting training of FC layer")
history = model.fit(train_dataset, validation_data=validation_dataset, epochs=5, callbacks=[AccuracyCallback("CNN_Transfer")])

# Descongelar capas finales
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# Finetune
print("Starting finetuning training")
history = model.fit(train_dataset, validation_data=validation_dataset, epochs=5, callbacks=[AccuracyCallback("CNN_Transfer")])
    

# Evaluar CNN
test_loss, test_acc = model.evaluate(validation_dataset)
print("\nCNN - Test accuracy:", test_acc)


Found 4752 files belonging to 9 classes.
Using 3802 files for training.
Found 4752 files belonging to 9 classes.
Using 950 files for validation.
Starting training of FC layer
Epoch 1/5


C:\Users\pablo\AppData\Local\Temp\ipykernel_34492\2647572324.py:19: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  self.mqtt_client = mqtt.Client()


119/119 ━━━━━━━━━━━━━━━━━━━━ 47s 380ms/step - accuracy: 0.4924 - loss: 1.4228 - val_accuracy: 0.6737 - val_loss: 0.9501
Epoch 2/5
119/119 ━━━━━━━━━━━━━━━━━━━━ 44s 373ms/step - accuracy: 0.6881 - loss: 0.8989 - val_accuracy: 0.7379 - val_loss: 0.8021
Epoch 3/5
119/119 ━━━━━━━━━━━━━━━━━━━━ 44s 372ms/step - accuracy: 0.7401 - loss: 0.7570 - val_accuracy: 0.7453 - val_loss: 0.7583
Epoch 4/5
119/119 ━━━━━━━━━━━━━━━━━━━━ 44s 372ms/step - accuracy: 0.7570 - loss: 0.6837 - val_accuracy: 0.7600 - val_loss: 0.7285
Epoch 5/5
119/119 ━━━━━━━━━━━━━━━━━━━━ 44s 371ms/step - accuracy: 0.7654 - loss: 0.6487 - val_accuracy: 0.7726 - val_loss: 0.7015
Starting finetuning training
Epoch 1/5
119/119 ━━━━━━━━━━━━━━━━━━━━ 58s 447ms/step - accuracy: 0.7138 - loss: 0.8394 - val_accuracy: 0.7674 - val_loss: 0.6976
Epoch 2/5
119/119 ━━━━━━━━━━━━━━━━━━━━ 53s 445ms/step - accuracy: 0.7688 - loss: 0.6627 - val_accuracy: 0.7832 - val_loss: 0.6786
Epoch 3/5
119/119 ━━━━━━━━━━━━━━━━━━━━ 53s 443ms/step - accuracy: 0.793

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2 
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


layers = tf.keras.layers
models = tf.keras.models
utils = tf.keras.utils

directory = 'datasets/realwaste-main/RealWaste/'

train_dataset = utils.image_dataset_from_directory(
    directory,
    labels="inferred",
    class_names=None,
    subset="training",
    validation_split=0.2,
    batch_size=8,
    image_size=(224,224),
    shuffle=True,
    seed=12,
)

validation_dataset = utils.image_dataset_from_directory(
    directory,
    labels="inferred",
    class_names=None,
    subset="validation",
    validation_split=0.2,
    batch_size=8,
    image_size=(224,224),
    shuffle=True,
    seed=12,
)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
])

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x , training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(9, activation="softmax")(x) 
model = models.Model(inputs, outputs)

model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy', # Sparce porque mis labels son enteros, sin sparce devuelve probabilidades
                  metrics=['accuracy'])

# Entrenar última capa
print("Starting training of FC layer")
history = model.fit(train_dataset, validation_data=validation_dataset, epochs=3, callbacks=[AccuracyCallback("CNN_Transfer")])

# Descongelar capas 
base_model.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# Finetune
print("Starting finetuning training")
history = model.fit(train_dataset, validation_data=validation_dataset, epochs=3, callbacks=[AccuracyCallback("CNN_Transfer")])
    

# Evaluar CNN
test_loss, test_acc = model.evaluate(validation_dataset)
print("\nCNN - Test accuracy:", test_acc)


### 1.2 Resnet50 con Transfer Learning

Dataset: RealWaste

Idea: Probar transfer learning descongelando distinto número de capas y observar el trade-off entre Accuracy y consumo. Mantener constante el entrenamiento de la FC.

(Potencialmente podría mirar variando las epochs de entrenamiento de la FC)

RESNET50

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50 
from tensorflow.keras.applications.resnet50 import preprocess_input


layers = tf.keras.layers
models = tf.keras.models
utils = tf.keras.utils

directory = 'datasets/realwaste-main/RealWaste/'

train_dataset = utils.image_dataset_from_directory(
    directory,
    labels="inferred",
    class_names=None,
    subset="training",
    validation_split=0.2,
    batch_size=32,
    image_size=(224,224),
    shuffle=True,
    seed=12,
)

validation_dataset = utils.image_dataset_from_directory(
    directory,
    labels="inferred",
    class_names=None,
    subset="validation",
    validation_split=0.2,
    batch_size=32,
    image_size=(224,224),
    shuffle=True,
    seed=12,
)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
])

resnet_model = ResNet50(
    weights = "imagenet",
    include_top=False, # eliminar la capa final para adaptarlo al dataset y categorizar como corresponda 
    input_shape=(224,224,3)
)

resnet_model.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = resnet_model(x , training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(9, activation="softmax")(x) 
model = models.Model(inputs, outputs)

model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# Entrenar última capa
print("Starting training of FC layer")
history = model.fit(train_dataset, validation_data=validation_dataset, epochs=5, callbacks=[AccuracyCallback("CNN_Transfer")])

# Descongelar capas finales
resnet_model.trainable = True
for layer in resnet_model.layers[:-10]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# Finetune
print("Starting finetuning training")
history = model.fit(train_dataset, validation_data=validation_dataset, epochs=5, callbacks=[AccuracyCallback("CNN_Transfer")])
    

# Evaluar CNN
test_loss, test_acc = model.evaluate(validation_dataset)
print("\nCNN - Test accuracy:", test_acc)


### 1.3 Validación visual

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

class_names = validation_dataset.class_names

for images, labels in validation_dataset.take(1):  # coge un batch
    images = images.numpy()
    labels = labels.numpy()

idxs = np.random.choice(len(images), 5, replace=False)

for i in idxs:
    img = images[i].astype("uint8")
    label = labels[i]

    img_input = np.expand_dims(images[i], axis=0)
    pred = np.argmax(model.predict(img_input, verbose=0))

    plt.imshow(img)
    plt.title(f"Pred: {class_names[pred]} | Real: {class_names[label]}")
    plt.axis("off")
    plt.show()

### 2. CNN propia

Dataset:

Idea: Probar una red convolucional utilizando diferente número de epochs y observando la relación entre el consumo y accuracy, tratando de determinar el final del periodo crítico y entender cuando deja de ser eficiente seguir entrenando. Posibilidad de quizás probar a reducir a partir de X punto (considerado el periodo crítico) la cantidad de recursos del entrenamiento para observar si la reducción de recursos impacta positivamente en el consumo sin producir un detrimento en el accuracy.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

layers = tf.keras.layers
models = tf.keras.models
utils = tf.keras.utils

directory = 'datasets/realwaste-main/RealWaste/'

train_dataset = utils.image_dataset_from_directory(
    directory,
    labels="inferred",
    class_names=None,
    subset="training",
    validation_split=0.2,
    batch_size=32,
    image_size=(256,256),
    shuffle=True,
    seed=12,
)

validation_dataset = utils.image_dataset_from_directory(
    directory,
    labels="inferred",
    class_names=None,
    subset="validation",
    validation_split=0.2,
    batch_size=32,
    image_size=(256,256),
    shuffle=True,
    seed=12,
)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
])

class_names = train_dataset.class_names
num_classes = len(class_names)

# Normalizar a [0,1]
normalization_layer = layers.Rescaling(1./255)
train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y))
validation_dataset = validation_dataset.map(lambda x, y: (normalization_layer(x), y))

# Definir modelo CNN
inputs = tf.keras.Input(shape=(256,256,3))
x = data_augmentation(inputs)
x = layers.Conv2D(16, (3,3), activation='relu')(x)
x = layers.MaxPooling2D((2,2))(x)
x = layers.Conv2D(32, (3,3), activation='relu')(x)
x = layers.MaxPooling2D((2,2))(x)
x = layers.Conv2D(64, (3,3), activation='relu')(x)
x = layers.MaxPooling2D((2,2))(x)
x = layers.Conv2D(128, (3,3), activation='relu')(x)
x = layers.MaxPooling2D((2,2))(x)
x = layers.Flatten()(x)
x = layers.Dense(64, activation='relu')(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# Entrenar CNN
history = model.fit(train_dataset, validation_data=validation_dataset, epochs=8, batch_size=64, callbacks=[AccuracyCallback("CNN_whole")])

# Evaluar CNN
test_loss, test_acc = model.evaluate(validation_dataset)
print("\nCNN - Test accuracy:", test_acc)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

for images, labels in validation_dataset.take(1):  # coge un batch
    images = images.numpy()
    labels = labels.numpy()

idxs = np.random.choice(len(images), 5, replace=False)

for i in idxs:
    img = (images[i] * 255).astype("uint8")
    label = labels[i]

    img_input = np.expand_dims(images[i], axis=0)
    pred = np.argmax(model.predict(img_input, verbose=0))

    plt.imshow(img)
    plt.title(f"Pred: {class_names[pred]} | Real: {class_names[label]}")
    plt.axis("off")
    plt.show()